## Imports

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import pytz

## Define the methods to:
* Fetch the tickers current prices, EPS and trainling PE
* Get the historic close prices to calculate change

In [2]:
def get_complete_stock_analysis(file_path):
    # 1. Load tickers
    df_tickers = pd.read_csv(file_path)
    tickers = df_tickers['Ticker'].unique().tolist()

    # 2. Batch download 1 year of data (Fastest for all technicals)
    print(f"Downloading historical data for {len(tickers)} tickers...")
    hist_data = yf.download(tickers, period="1y", interval="1d", progress=False)

    results = []

    for ticker in tickers:
        try:
            # Fetch Ticker object for P/E Ratio (Fundamental)
            t_obj = yf.Ticker(ticker)
            info = t_obj.info

            # Extract price series for this ticker
            ticker_hist = hist_data.xs(ticker, axis=1, level=1).dropna()

            if len(ticker_hist) < 2:
                continue

            close_prices = ticker_hist['Close']
            high_prices = ticker_hist['High']

            current_price = close_prices.iloc[-1]
            prev_close = close_prices.iloc[-2]

            # --- 1. NEW: Daily % Change (Current vs Last Closure) ---
            daily_pct_change = ((current_price - prev_close) / prev_close) * 100

            # --- 2. % Change (1 Month) ---
            price_1m_ago = close_prices.iloc[-22] if len(close_prices) >= 22 else close_prices.iloc[0]
            pct_change_1m = ((current_price - price_1m_ago) / price_1m_ago) * 100

            # --- 3. % Off 52-Week High ---
            high_52w = high_prices.max()
            pct_off_52w_high = ((current_price - high_52w) / high_52w) * 100

            # --- 4. RSI (14-Day) ---
            delta = close_prices.diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()

            if loss.iloc[-1] == 0:
                rsi = 100 if gain.iloc[-1] > 0 else 50
            else:
                rs = gain / loss
                rsi = 100 - (100 / (1 + rs)).iloc[-1]

            # --- 5. Fundamental: P/E Ratio ---
            pe_ratio = info.get('trailingPE') or info.get('forwardPE')

            # --- 6. Sector ---
            sector = df_tickers[df_tickers.Ticker == ticker].Sector.iloc[0]

            # --- 7. Currency ---
            currency = df_tickers[df_tickers.Ticker == ticker].Currency.iloc[0]

            results.append({
                'Ticker': ticker,
                'Price': round(current_price, 2),
                '% 1D': f"{daily_pct_change:.1f}%",
                '% 1M': f"{pct_change_1m:.1f}%",
                '% 52W H': f"{pct_off_52w_high:.1f}%",
                'RSI': round(rsi, 1),
                'PE': round(pe_ratio, 1) if pe_ratio else "N/A",
                'Sector': sector,
                'Currency': currency
            })

        except Exception as e:
            print(f"Error processing {ticker}: {e}")

    # Create and save the final report
    df_final = pd.DataFrame(results)
    #df_final.to_csv('final_stock_report.csv', index=False)
    return df_final

### Fetch price history and calculate metrics

In [3]:
# Execute the update
analysis_results = get_complete_stock_analysis('dividend.csv')
print(analysis_results.head())

     Ticker   Price   % 1D  % 1M % 52W H   RSI    PE Sector Currency
0  NORCO.OL   40.45  -0.1%  0.4%  -18.4%  39.7  19.0    IND      NOK
1   ATEA.OL  143.60   0.1%  4.5%  -12.3%  58.9  18.6    TEC      NOK
2    DNB.OL  279.60  -0.2%  1.0%   -5.3%  40.4  10.0    FIN      NOK
3   NONG.OL  155.78   0.3%  3.7%   -4.6%  47.7  10.2    FIN      NOK
4    STB.OL  168.90  -0.4%  2.4%   -5.0%  46.1  14.4    FIN      NOK


### Sort and format dataframe

In [4]:
df_sorted = analysis_results.sort_values(by = ["Sector", "PE"])
print(df_sorted)

      Ticker   Price   % 1D   % 1M % 52W H   RSI    PE Sector Currency
15  MORLD.OL   18.98   2.3%  14.9%   -3.3%  81.1  11.9    ENE      NOK
16    ODL.OL   98.40  -0.1%   1.4%  -10.3%  43.5  15.3    ENE      NOK
7     ENH.OL    8.84   0.5%   1.6%  -13.8%  44.5  23.9    ENE      NOK
2     DNB.OL  279.60  -0.2%   1.0%   -5.3%  40.4  10.0    FIN      NOK
3    NONG.OL  155.78   0.3%   3.7%   -4.6%  47.7  10.2    FIN      NOK
10   MING.OL  205.20  -0.1%   5.1%   -4.2%  49.1  10.7    FIN      NOK
19      ITUB    8.81  -2.3%   9.5%   -7.9%  63.0  11.0    FIN      USD
11  SB1NO.OL  197.00   0.0%   1.5%   -5.6%  46.8  11.6    FIN      NOK
4     STB.OL  168.90  -0.4%   2.4%   -5.0%  46.1  14.4    FIN      NOK
9     B2I.OL   24.80   0.0%   6.7%   -3.1%  61.5  14.8    FIN      NOK
8     GJF.OL  260.20   0.2%   5.2%   -9.0%  59.7  20.7    FIN      NOK
13   MPCC.OL   21.47   1.1%  -3.0%  -10.9%  35.9   4.3    IND      NOK
12    WWI.OL  693.00   0.0%  -5.5%  -11.5%  30.4   4.8    IND      NOK
6    W

### Format HTML report and save

In [5]:
# Create the table HTML string
table_html = df_sorted.to_html(
    index=False, 
    classes='table table-striped table-hover table-sm', # Bootstrap classes
    border=0,
    justify='unset'
)

In [6]:
def color_change(val):
    color = 'green' if val > 0 else 'red'
    return f'color: {color}; font-weight: bold'

In [7]:
styled_html = (
    df_sorted.style
    .format({
        'Price': '{:.2f}',      # Force 2 decimals
        'PE': '{:.1f}',      # Force 1 decimal
        'RSI': '{:+.1f}'   # Force 1 decimal, a '+' sign for pos numbers, and a % symbol
    })
    .set_table_attributes('class="table table-striped table-hover"')
    .hide(axis='index')
    .to_html()
)

In [8]:
# Find local time
oslo_tz = pytz.timezone("Europe/Oslo")
oslo_time = pd.Timestamp.now(oslo_tz).strftime('%Y-%m-%d %H:%M')

# Inject styled_html into your template
html_template = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sector Report</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <style>
        body {{ padding: 5px; background-color: #f8f9fa; }}
        .table-responsive {{ border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); background: white; margin: 0;}}
        h2 {{ color: #343a40; margin-bottom: 20px; font-weight: bold; }}
        /* Ensure the pandas-generated styles don't conflict with Bootstrap */
        table {{ width: 100%; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Sector Performance Report</h2>
        <div class="table-responsive">
            {styled_html}
        </div>
        <p class="text-muted mt-3">Last updated: {oslo_time}</p>
    </div>
</body>
</html>
"""

# Save the full report
with open("stock_report.html", "w") as f:
    f.write(html_template)